# Process Existing Data for GNN Cross-Network Prediction

This notebook combines multiple recordings from each session into a single file
that can be used by `gnn_cross_network_prediction.py`.

**Current data structure:**
```
LIF data/
├── 20260105_133915/
│   ├── network_20260105_133915.npz
│   ├── recording000.npz ... recording019.npz
│   └── session_metadata.json
```

**Output:**
- `recording_combined.npz` - all recordings merged with time offsets
- `session_gnn_metadata.json` - metadata file for GNN script

In [5]:
import numpy as np
import json
import os
import glob
from pathlib import Path

In [6]:
# Configuration - path to LIF data folder
DATA_PATH = "LIF data"

# Verify path exists
print(f"Data path exists: {os.path.exists(DATA_PATH)}")
if os.path.exists(DATA_PATH):
    folders = [f for f in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, f))]
    print(f"Number of session folders: {len(folders)}")
    for f in sorted(folders):
        print(f"  - {f}")

Data path exists: True
Number of session folders: 5
  - 20260223_130621
  - 20260223_151236
  - 20260223_172834
  - 20260223_193040
  - 20260223_215559


In [7]:
def find_all_sessions(data_path):
    """
    Find all session folders containing session_metadata.json.
    
    Structure: LIF data/{timestamp}/session_metadata.json
    """
    metadata_files = sorted(glob.glob(f"{data_path}/*/session_metadata.json"))
    return metadata_files


def load_session_data(metadata_file):
    """
    Load session metadata and network data.
    
    Returns dict with metadata, network data, and session directory.
    """
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    session_dir = os.path.dirname(metadata_file)
    
    # Load network data
    network_file = os.path.join(session_dir, f"network_{metadata['timestamp']}.npz")
    network_data = dict(np.load(network_file, allow_pickle=True))
    
    return {
        'metadata': metadata,
        'network': network_data,
        'session_dir': session_dir,
        'timestamp': metadata['timestamp']
    }


def combine_recordings(session_data):
    """
    Combine all recordings from a session into one spike_times array
    and concatenate voltage traces along the time axis.
    
    Each recording's spike times are offset by (recording_index * recording_duration)
    so they form a continuous timeline. Voltage traces are concatenated column-wise.
    
    Returns:
        combined_spike_times: list of arrays, one per neuron
        total_duration: total duration in ms
        combined_voltage: dict with 'traces', 'times', 'sample_rate' or None
    """
    metadata = session_data['metadata']
    session_dir = session_data['session_dir']
    recording_duration = metadata['recording_duration']
    
    # Get successful recordings
    recordings = [r for r in metadata['recordings'] if r.get('success', False)]
    print(f"  Found {len(recordings)} successful recordings")
    
    if len(recordings) == 0:
        raise ValueError("No successful recordings found")
    
    # Determine number of neurons from first recording
    first_rec_file = os.path.join(session_dir, os.path.basename(recordings[0]['file']))
    first_rec = np.load(first_rec_file, allow_pickle=True)
    n_neurons = len(first_rec['spike_times'])
    print(f"  Number of neurons: {n_neurons}")
    
    # Check if voltage data is available
    has_voltage = 'voltage_traces' in first_rec
    if has_voltage:
        voltage_sample_rate = float(first_rec['voltage_sample_rate'])
        print(f"  Voltage traces found (sample rate: {voltage_sample_rate} ms)")
    else:
        print(f"  No voltage traces in recordings")
    
    # Initialize combined spike times
    combined_spike_times = [[] for _ in range(n_neurons)]
    voltage_trace_segments = []  # list of (n_neurons, n_samples) arrays
    
    # Process each recording
    for rec_info in recordings:
        rec_index = rec_info['index']
        rec_file = os.path.join(session_dir, os.path.basename(rec_info['file']))
        
        if not os.path.exists(rec_file):
            print(f"  Warning: Recording file not found: {rec_file}")
            continue
        
        rec_data = np.load(rec_file, allow_pickle=True)
        spike_times = rec_data['spike_times']
        
        # Time offset for this recording
        time_offset = rec_index * recording_duration
        
        # Add spike times with offset
        for neuron_idx in range(n_neurons):
            neuron_spikes = spike_times[neuron_idx]
            if len(neuron_spikes) > 0:
                offset_spikes = np.array(neuron_spikes) + time_offset
                combined_spike_times[neuron_idx].extend(offset_spikes.tolist())
        
        # Collect voltage traces for concatenation
        if has_voltage and 'voltage_traces' in rec_data:
            voltage_trace_segments.append(rec_data['voltage_traces'])
    
    # Convert to sorted numpy arrays
    for i in range(n_neurons):
        combined_spike_times[i] = np.array(sorted(combined_spike_times[i]))
    
    total_duration = len(recordings) * recording_duration
    total_spikes = sum(len(st) for st in combined_spike_times)
    print(f"  Total duration: {total_duration} ms ({total_duration/1000:.1f} s)")
    print(f"  Total spikes: {total_spikes}")
    
    # Combine voltage traces
    combined_voltage = None
    if has_voltage and len(voltage_trace_segments) > 0:
        combined_traces = np.concatenate(voltage_trace_segments, axis=1)
        combined_times = np.arange(combined_traces.shape[1]) * voltage_sample_rate
        combined_voltage = {
            'traces': combined_traces,
            'times': combined_times,
            'sample_rate': voltage_sample_rate
        }
        print(f"  Combined voltage: {combined_traces.shape[0]} neurons x {combined_traces.shape[1]} time points")
    
    return combined_spike_times, total_duration, combined_voltage


def save_combined_for_gnn(session_data, combined_spike_times, total_duration, combined_voltage=None):
    """
    Save combined recording and create GNN-compatible metadata file.
    
    Creates:
        - recording_combined.npz in session folder (spikes + voltage traces)
        - session_gnn_metadata.json in session folder (for GNN script)
    """
    session_dir = session_data['session_dir']
    timestamp = session_data['timestamp']
    metadata = session_data['metadata']
    
    # Build save dictionary
    save_dict = {
        'spike_times': np.array(combined_spike_times, dtype=object),
        'duration_ms': total_duration,
        'n_neurons': len(combined_spike_times)
    }
    
    # Include voltage traces if available
    if combined_voltage is not None:
        save_dict['voltage_traces'] = combined_voltage['traces']
        save_dict['voltage_times'] = combined_voltage['times']
        save_dict['voltage_sample_rate'] = combined_voltage['sample_rate']
        print(f"  Including voltage traces: {combined_voltage['traces'].shape}")
    
    # Save combined recording
    combined_file = os.path.join(session_dir, "recording_combined.npz")
    np.savez_compressed(combined_file, **save_dict)
    print(f"  Saved: {combined_file}")
    
    # Create GNN-compatible metadata
    # Use relative paths from the main working directory
    folder_name = os.path.basename(session_dir)
    
    gnn_metadata = {
        'timestamp': timestamp,
        'n_recordings_combined': len([r for r in metadata['recordings'] if r.get('success')]),
        'recording_duration': int(total_duration),
        'num_clusters': metadata.get('num_clusters', 0),
        'num_neurons': len(combined_spike_times),
        'num_connections': metadata.get('num_connections', 0),
        'network_file': f"LIF data/{folder_name}/network_{timestamp}.npz",
        'recordings': [{
            'index': 0,
            'file': f"LIF data/{folder_name}/recording_combined.npz",
            'success': True,
            'num_spikes': int(sum(len(st) for st in combined_spike_times))
        }]
    }
    
    gnn_metadata_file = os.path.join(session_dir, "session_gnn_metadata.json")
    with open(gnn_metadata_file, 'w') as f:
        json.dump(gnn_metadata, f, indent=2)
    print(f"  Saved: {gnn_metadata_file}")
    
    return gnn_metadata_file

## Find Available Sessions

In [8]:
# Find all session metadata files
metadata_files = find_all_sessions(DATA_PATH)
print(f"Found {len(metadata_files)} sessions:\n")

for i, mf in enumerate(metadata_files):
    with open(mf, 'r') as f:
        meta = json.load(f)
    
    timestamp = meta.get('timestamp', 'unknown')
    n_recordings = len([r for r in meta.get('recordings', []) if r.get('success')])
    n_neurons = meta.get('num_neurons', '?')
    duration = meta.get('recording_duration', 0)
    
    # Check if already has GNN-compatible file
    session_dir = os.path.dirname(mf)
    combined_file = os.path.join(session_dir, "recording_combined.npz")
    has_gnn = os.path.exists(combined_file)
    status = "GNN-ready" if has_gnn else "Needs processing"
    
    total_time = n_recordings * duration / 1000  # seconds
    print(f"{i}: {timestamp} - {n_recordings} recordings ({total_time:.0f}s), {n_neurons} neurons [{status}]")

Found 5 sessions:

0: 20260223_130621 - 5 recordings (300s), 299 neurons [Needs processing]
1: 20260223_151236 - 5 recordings (300s), 307 neurons [Needs processing]
2: 20260223_172834 - 5 recordings (300s), 294 neurons [Needs processing]
3: 20260223_193040 - 5 recordings (300s), 314 neurons [Needs processing]
4: 20260223_215559 - 5 recordings (300s), 287 neurons [Needs processing]


## Process All Sessions

In [9]:
def process_all_sessions(data_path, skip_existing=True):
    """
    Process all sessions to create GNN-compatible combined files.
    
    Args:
        data_path: Path to LIF data folder
        skip_existing: If True, skip sessions that already have combined files
    """
    metadata_files = find_all_sessions(data_path)
    print(f"Found {len(metadata_files)} sessions\n")
    
    processed, skipped, failed = [], [], []
    
    for mf in metadata_files:
        session_dir = os.path.dirname(mf)
        with open(mf, 'r') as f:
            meta = json.load(f)
        timestamp = meta.get('timestamp')
        
        # Check if already processed
        combined_file = os.path.join(session_dir, "recording_combined.npz")
        if skip_existing and os.path.exists(combined_file):
            print(f"Skipping {timestamp} - already has combined file")
            skipped.append(timestamp)
            continue
        
        print(f"\nProcessing: {timestamp}")
        try:
            session_data = load_session_data(mf)
            combined_spikes, total_duration, combined_voltage = combine_recordings(session_data)
            save_combined_for_gnn(session_data, combined_spikes, total_duration, combined_voltage)
            processed.append(timestamp)
            print(f"  SUCCESS")
        except Exception as e:
            print(f"  ERROR: {e}")
            failed.append((timestamp, str(e)))
    
    print(f"\n{'='*60}")
    print(f"Summary: Processed={len(processed)}, Skipped={len(skipped)}, Failed={len(failed)}")
    
    if failed:
        print(f"\nFailed sessions:")
        for ts, err in failed:
            print(f"  {ts}: {err}")
    
    return processed, skipped, failed


# Run processing
processed, skipped, failed = process_all_sessions(DATA_PATH, skip_existing=True)

Found 5 sessions


Processing: 20260223_130621
  Found 5 successful recordings
  Number of neurons: 299
  Voltage traces found (sample rate: 1.0 ms)
  Total duration: 300000 ms (300.0 s)
  Total spikes: 41392
  Combined voltage: 299 neurons x 300000 time points
  Including voltage traces: (299, 300000)
  Saved: LIF data\20260223_130621\recording_combined.npz
  Saved: LIF data\20260223_130621\session_gnn_metadata.json
  SUCCESS

Processing: 20260223_151236
  Found 5 successful recordings
  Number of neurons: 307
  Voltage traces found (sample rate: 1.0 ms)
  Total duration: 300000 ms (300.0 s)
  Total spikes: 28588
  Combined voltage: 307 neurons x 300000 time points
  Including voltage traces: (307, 300000)
  Saved: LIF data\20260223_151236\recording_combined.npz
  Saved: LIF data\20260223_151236\session_gnn_metadata.json
  SUCCESS

Processing: 20260223_172834
  Found 5 successful recordings
  Number of neurons: 294
  Voltage traces found (sample rate: 1.0 ms)
  Total duration: 300000 

## Verify GNN Compatibility

In [10]:
def test_gnn_loader(gnn_metadata_file):
    """
    Test loading data exactly like gnn_cross_network_prediction.py does.
    
    This mimics the load_network() function in the GNN script.
    """
    print(f"Testing: {gnn_metadata_file}\n")
    
    with open(gnn_metadata_file) as f:
        metadata = json.load(f)
    
    # Load network (using path from metadata)
    network_data = np.load(metadata['network_file'], allow_pickle=True)
    
    # Load recording (using path from metadata)
    rec_file = metadata['recordings'][0]['file']
    rec_data = np.load(rec_file, allow_pickle=True)
    
    # Validate
    connections = network_data['connections']
    positions = network_data['neuron_positions']
    spike_times = rec_data['spike_times']
    duration = metadata['recording_duration']
    
    print(f"  Network: {len(connections)} connections, {len(positions)} neurons")
    print(f"  Recording: {len(spike_times)} neurons with spike data")
    print(f"  Duration: {duration} ms ({duration/1000:.1f} s)")
    
    total_spikes = sum(len(st) for st in spike_times)
    avg_rate = total_spikes / len(spike_times) / (duration/1000)
    print(f"  Total spikes: {total_spikes} (avg {avg_rate:.1f} Hz per neuron)")
    
    # Check position-neuron consistency
    if len(positions) == len(spike_times):
        print(f"  Neuron count matches between network and recording")
    else:
        print(f"  MISMATCH: {len(positions)} positions vs {len(spike_times)} spike trains")
        return False
    
    print("  SUCCESS: Data is GNN-compatible!")
    return True


# Test all GNN metadata files
gnn_metadata_files = sorted(glob.glob(f"{DATA_PATH}/*/session_gnn_metadata.json"))
print(f"Found {len(gnn_metadata_files)} GNN-ready sessions\n")
print("="*60 + "\n")

for gnn_mf in gnn_metadata_files:
    test_gnn_loader(gnn_mf)
    print("\n" + "-"*60 + "\n")

Found 5 GNN-ready sessions


Testing: LIF data\20260223_130621\session_gnn_metadata.json

  Network: 2808 connections, 299 neurons
  Recording: 299 neurons with spike data
  Duration: 300000 ms (300.0 s)
  Total spikes: 41392 (avg 0.5 Hz per neuron)
  Neuron count matches between network and recording
  SUCCESS: Data is GNN-compatible!

------------------------------------------------------------

Testing: LIF data\20260223_151236\session_gnn_metadata.json

  Network: 3101 connections, 307 neurons
  Recording: 307 neurons with spike data
  Duration: 300000 ms (300.0 s)
  Total spikes: 28588 (avg 0.3 Hz per neuron)
  Neuron count matches between network and recording
  SUCCESS: Data is GNN-compatible!

------------------------------------------------------------

Testing: LIF data\20260223_172834\session_gnn_metadata.json

  Network: 2758 connections, 294 neurons
  Recording: 294 neurons with spike data
  Duration: 300000 ms (300.0 s)
  Total spikes: 27084 (avg 0.3 Hz per neuron)
  Neur

## GNN Script Usage

After running this notebook, update `gnn_cross_network_prediction.py` line 1406:

```python
# Change from:
metadata_files = sorted(glob.glob('LIF data/session_*_metadata.json'))

# To:
metadata_files = sorted(glob.glob('LIF data/*/session_gnn_metadata.json'))
```

In [11]:
# List all GNN metadata files
gnn_metadata_files = sorted(glob.glob(f"{DATA_PATH}/*/session_gnn_metadata.json"))

print("GNN metadata files ready for use:")
print("="*60)
for f in gnn_metadata_files:
    print(f"  {f}")

print(f"\nTotal: {len(gnn_metadata_files)} sessions ready for GNN training")

GNN metadata files ready for use:
  LIF data\20260223_130621\session_gnn_metadata.json
  LIF data\20260223_151236\session_gnn_metadata.json
  LIF data\20260223_172834\session_gnn_metadata.json
  LIF data\20260223_193040\session_gnn_metadata.json
  LIF data\20260223_215559\session_gnn_metadata.json

Total: 5 sessions ready for GNN training
